In [ ]:
import pickle 
import torch

with open("/home/xujiaming/xujiaming/Paper/SpecKV/SpecKV/attn_data/ori_attn.pkl","rb") as f:
    ori_attn = pickle.load(f)
with open("/home/xujiaming/xujiaming/Paper/SpecKV/SpecKV/attn_data/spec_attn.pkl","rb") as f:
    spec_attn = pickle.load(f)


In [4]:
from models.Ori_qwen_model import Qwen3ForCausalLM

You are using a model of type qwen3 to instantiate a model of type llama. This is not supported for all configurations of models and can yield errors.
LlamaForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Loading checkpoint shards: 100%|██████████| 5/5 [00:04<00:00,  1.10it/s]
Some weights of LlamaForCausalLM were not initialized from the model checkpoint at /share/public/public_models/Qwen3-8B and are newly initialized: ['model.layers.0.self_attn.rotary_emb.inv_freq', 'model.layers.1.self_attn.rotary_emb.inv_freq', 'model.layers.10.self_attn.rotary_emb.inv_freq', 'model.layers.11.self_attn.rotary_emb.inv_freq', 'model.layers.12.self_attn.rotary_emb.inv_freq', 'model.layers.13.self_attn.rotary_emb.inv_freq', 'model.layers.14.self_attn.rotary_emb.inv_freq', 'model.layers.15.self_attn.rotary_emb.inv_freq', 'model.layers.16.self_attn.rotary_emb.inv_freq', 'model.layers.17.self_attn.rotary_emb.inv_freq', 'model.layers.18.self_attn.rotary_emb.inv_freq', 'model.layers.19.self_attn.rotary_emb.inv_freq', 'model.layers.2.self_attn.rotary_emb.inv_freq', 'model.layers.20.self_attn.rotary_emb.inv_freq', 'model.layers.21.self_attn.rotary_emb.inv_freq', 'model.layers.22.self_attn.rotary_emb.inv_freq'

In [1]:
from models.Ori_qwen_model import Qwen3ForCausalLM
ori_model = Qwen3ForCausalLM.from_pretrained("/share/public/public_models/Qwen3-8B")

/home/xujiaming/xujiaming/anaconda3/envs/speckv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 5/5 [00:04<00:00,  1.24it/s]


In [2]:
ori_model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
        (post_attention_layernorm): 

In [ ]:
len(spec_attn)

In [ ]:
spec_attn[0].shape

In [ ]:
ori_top_value, ori_top_index = torch.topk(ori_attn[0], 10, dim=-1)
spec_top_value, spec_top_index = torch.topk(spec_attn[0], int(spec_attn[0].shape[-1]*0.2), dim=-1)

In [ ]:
value = torch.rand(1,32,158, 128).cuda()

In [ ]:
spec_top_index = spec_top_index.squeeze(2).unsqueeze(-1)

In [ ]:
spec_top_index = spec_top_index.expand(-1, -1, -1, value.shape[-1])

In [ ]:
spec_top_index[:,0]

In [ ]:
tmp = torch.gather(value, dim=2, index = spec_top_index)

In [ ]:
value[:,0,157]
for i in range(value.shape[1]):
    for j in range(spec_top_index.shape[2]):
        if False in value[:,i,spec_top_index[0,i,j,0]] == tmp[:,i,j]:
            print("wrong")

In [ ]:
tmp[:,0]

In [ ]:
tmp = torch.gather(ori_attn[0][0:1], dim=-1, index=spec_top_index[0:1])
print(tmp.shape)

In [ ]:
spec_top_index = spec_top_index.transpose(-1,-2)

In [ ]:
spec_top_index.shape

In [ ]:
tmp = torch.gather(value, dim=2, index=spec_top_index)
print(tmp.shape)

In [ ]:
spec_top_index.dtype

In [ ]:
torch.cat([spec_top_index,torch.zeros_like(spec_top_index[...,:1]).cuda()],dim=-1)

In [ ]:
a[spec_top_index].shape

In [ ]:
ori_attn_value = ori_attn[0][:,0,0,spec_top_index]

In [ ]:
ori_attn_value=ori_attn_value.squeeze()

In [ ]:
torch.sum(ori_attn_value,dim=-1)

In [ ]:
torch.sum(ori_attn_value[0])

In [ ]:
torch.unique(torch.zeros(1,10))

In [ ]:
tokenizer.decode([  791,  6864,  3224,  3363,  7928, 18880,  4033,  5426,  4320,  1925,
          6156])

In [ ]:
tokenizer.decode([128000])

In [ ]:
tokenizer.encode("The capital of China is Beijing.")